# E03 — Marine ageing and strength-loss kinetics
Reproduce the committed exploratory model comparison using the 71-record E02 core. Baselines are fixed; time-point holdout is not independent-specimen validation. The Python notebook is a runnable translation of the executed comparison; it checks numerical agreement with saved predictions.


In [ ]:
from pathlib import Path
import csv, math
from collections import defaultdict
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'data/E02/mechanical_ageing_core_v0.0.1.csv').exists():
    if (ROOT.parent / 'data/E02/mechanical_ageing_core_v0.0.1.csv').exists():
        ROOT = ROOT.parent
    else:
        import subprocess
        ROOT = Path('/content/SeaweedMat')
        if not ROOT.exists():
            subprocess.run(['git','clone','https://github.com/peterDataScientia/SeaweedMat.git',str(ROOT)],check=True)
INPUT = ROOT / 'data/E02/mechanical_ageing_core_v0.0.1.csv'
OUT = ROOT / 'outputs/E03_rerun'
OUT.mkdir(parents=True, exist_ok=True)
rows = list(csv.DictReader(INPUT.open()))
assert len(rows) == 71
by_id = {r['record_id']:r for r in rows}
assert len(by_id) == len(rows)
groups = defaultdict(list)
for r in rows:
    if float(r['exposure_days_nominal']) == 0:
        continue
    key = '|'.join((r['baseline_id'],r['condition'],r['temperature_C']))
    groups[key].append(r)
assert len(groups) == 16
models = ['constant','zero_order','first_order','weibull','lag_first_order']


## Models and deterministic optimisation
Rates are fit on training-scaled time. Profile grids restrict Weibull shape to 0.25–4 and lag to 0–0.8 of the largest training time; these are modelling choices. Fit only positive-time observations. Above-100 observations are retained; monotone models cannot represent strengthening.


In [ ]:
def predict(model,t,k=0.,p=1.):
    if model == 'constant': return 100.
    if model == 'zero_order': return 100*max(0,1-k*t)
    if model == 'first_order': return 100*math.exp(-k*t)
    if model == 'weibull': return 100*math.exp(-k*t**p)
    return 100*math.exp(-k*max(0,t-p))

def golden(fun,a,b):
    ratio=.618033988749895
    c=b-(b-a)*ratio; d=a+(b-a)*ratio
    fc,fd=fun(c),fun(d)
    for _ in range(55):
        if fc<fd:
            b,d,fd=d,c,fc
            c=b-(b-a)*ratio; fc=fun(c)
        else:
            a,c,fc=c,d,fd
            d=a+(b-a)*ratio; fd=fun(d)
    return (a+b)/2

def fit(model,train):
    scale=max(t for t,y,rid in train)
    xy=[(t/scale,y) for t,y,rid in train]
    def objective(k,p): return sum((predict(model,t,k,p)-y)**2 for t,y in xy)
    p0=0. if model=='lag_first_order' else 1.
    best=(0.,p0,objective(0,p0),scale)
    if model=='constant': return best
    shapes=[.25+i*.05 for i in range(76)] if model=='weibull' else [i*.01 for i in range(81)] if model=='lag_first_order' else [1.]
    max_k=5 if model=='zero_order' else 30
    grid=[i*max_k/80 for i in range(81)]
    for p in shapes:
        vals=[objective(k,p) for k in grid]
        i=min(range(len(grid)),key=vals.__getitem__)
        k=golden(lambda v:objective(v,p),grid[max(0,i-1)],grid[min(80,i+1)])
        for cand in [k,grid[i],0,max_k]:
            loss=objective(cand,p)
            if loss<best[2]-1e-10: best=(cand,p,loss,scale)
    return best


In [ ]:
predictions=[]; params=[]; series=[]
for sid,rr in groups.items():
    rr=sorted(rr,key=lambda r:float(r['exposure_days_nominal']))
    assert float(by_id[rr[0]['baseline_id']]['retention_pct'])==100
    pts=[(float(r['exposure_days_nominal']),float(r['retention_pct']),r['record_id']) for r in rr]
    n=len(pts); exact=rr[0]['study']=='ArayaSchmidt_Queirolo_2019'
    st='exact_table_primary' if exact else 'graph_common_supplementary' if n>=4 else 'graph_simple_supplementary' if n>=3 else 'sparse_descriptive'
    eligible=models if n>=4 else models[:3] if n>=3 else []
    label=rr[0]['configuration'].split(' / ')[0] if exact else 'Zhou '+rr[0]['material'] if rr[0]['study']=='Zhou_2019' else 'Rayon '+('biotic' if rr[0]['condition'].startswith('renewed') else 'abiotic') if rr[0]['study']=='Chamley_2025' else 'PBS/PBAT '+rr[0]['temperature_C']+' C'
    series.append(dict(series_id=sid,label=label,stratum=st,n_postbaseline=n,eligible_models=';'.join(eligible)))
    def add(model,i,hat):
        t,y,rid=pts[i]
        predictions.append(dict(series_id=sid,label=label,stratum=st,model=model,record_id=rid,heldout_day=t,observed_pct=y,predicted_pct=round(hat,6),error_pp=round(hat-y,6),split='terminal_forecast' if i==n-1 else 'interior_holdout'))
    for model in eligible:
        k,p,sse,scale=fit(model,pts)
        params.append(dict(series_id=sid,label=label,stratum=st,model=model,rate_scaled=k,shape_or_lag_scaled=p,time_scale_days=scale,train_rmse_pp=math.sqrt(sse/n)))
        for i,(t,y,rid) in enumerate(pts):
            k,p,sse,scale=fit(model,pts[:i]+pts[i+1:])
            add(model,i,predict(model,t/scale,k,p))
    if eligible:
        for i in range(n-1):
            lt,ly,_=pts[i-1] if i else (0,100,'baseline')
            ht,hy,_=pts[i+1]
            t,y,_=pts[i]
            add('linear_interpolation',i,ly+(hy-ly)*(t-lt)/(ht-lt))
        add('last_observation',n-1,pts[-2][1])
P=pd.DataFrame(predictions)
F=pd.DataFrame(params)
S=pd.DataFrame(series)
assert len(P)==294 and len(F)==59
M=P.groupby(['stratum','split','model']).agg(n_predictions=('error_pp','size'),n_series=('series_id','nunique'),MAE_pp=('error_pp',lambda x:np.mean(np.abs(x))),RMSE_pp=('error_pp',lambda x:np.sqrt(np.mean(x*x))),bias_pp=('error_pp','mean')).reset_index()
for name,frame in [('heldout_predictions',P),('model_parameters',F),('series_audit',S),('benchmark_metrics',M)]:
    frame.to_csv(OUT/(name+'.csv'),index=False)
display(M[M.stratum=='exact_table_primary'].sort_values(['split','RMSE_pp']))

# Independently compare this rerun with the committed calculation.
ref=pd.read_csv(ROOT/'outputs/E03/heldout_predictions.csv')
keys=['series_id','model','record_id','split']
J=P.merge(ref,on=keys,validate='one_to_one',suffixes=('_rerun','_committed'))
assert len(J)==len(P)==len(ref)
max_diff=(J.predicted_pct_rerun-J.predicted_pct_committed).abs().max()
print('Maximum difference from committed predictions (percentage points):',max_diff)
assert max_diff<0.001


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'DejaVu Sans','font.weight':'bold','axes.labelweight':'bold','axes.labelsize':15,'font.size':12,'axes.linewidth':2,'xtick.major.width':2,'ytick.major.width':2,'xtick.major.size':6,'ytick.major.size':6,'axes.spines.top':False,'axes.spines.right':False,'legend.fontsize':11})
p=M[M.stratum=='exact_table_primary'].pivot(index='model',columns='split',values='RMSE_pp').reindex(models+['linear_interpolation','last_observation'])
p.index=['Constant','Zero-order','First-order','Weibull','Lag-phase','Interpolation','Last observation']
fig,ax=plt.subplots(figsize=(160/25.4,125/25.4))
p.plot.barh(ax=ax,color=['#075DE8','#008D79'],width=.8)
ax.set_xlabel('RMSE [percentage points]')
ax.set_ylabel('')
ax.invert_yaxis()
ax.legend(['Interior holdout','Terminal forecast'],frameon=False)
fig.tight_layout()
for ext in ['png','svg','pdf']:
    fig.savefig(OUT/('kinetic_benchmark.'+ext),dpi=600,bbox_inches='tight',facecolor='white')
plt.show()
